# Phase 1: MVP, a coding agent that can fix a bug safely

Phase 0's agent can only **read**. It can explain a bug but not fix it. This phase gives it the tools to explore, edit and run code, with safety checks before anything risky.

This notebook is the lab: each step's plan, evidence, code and observations live here. [phase_1_README.md](phase_1_README.md) is written once the phase is finished (see [WORKFLOW.md](../WORKFLOW.md)).

## Goal

**Done when:** given a small project with a failing test, the agent

1. finds the bug,
2. fixes it,
3. runs the tests to confirm the fix,
4. and asks before anything risky.

## Plan

One section per roadmap step, built strictly in this order. Each step starts from what already exists and changes only what it needs.

| # | Section | What it adds | Why at this point |
|---|---|---|---|
| 1 | Tool registry | One place that lists the tools; the prompt, parser and loop read from it | Set up before adding tools, so each new tool is a function, not a prompt rewrite |
| 2 | List and find files | Tools to see what's in a project | The agent has to explore a project before it can change it |
| 3 | Folder boundary | File tools refuse paths outside the project | Before any tool can write |
| 4 | Write and edit files | Tools that change files | The moment the agent can actually change code |
| 5 | Approval prompt | Ask the user before risky actions | Needed before shell access |
| 6 | Run shell commands | A tool that runs commands | Lets the agent run code, tests and git |
| 7 | Verify loop | Run the tests, fix what fails, repeat | What makes it a *coding* agent |

After section 7: an end-to-end demo against the goal above. Then this notebook's code is refactored into `agent_v1/` (see [WORKFLOW.md](../WORKFLOW.md)).

## How this notebook works

- It imports **only** Phase 0's frozen package, `agent_v0`. Nothing comes from `agent_v1`, which doesn't exist yet.
- Everything new is built here. To change a Phase 0 function, we look at the imported version, then write a new one **with the same name** in this notebook (*shadowing*). We never modify `agent_v0` itself.
- **Every step starts with a game plan**: the problem, how real agents solve it, the proposed sub-steps, decisions and acceptance criteria. We agree on it before any code, then build the sub-steps one at a time.

## 0. Setup: import Phase 0

`phase-0-foundation` has a hyphen in its name, so Python can't import it as a package. Instead we add the folder to Python's search path, then import `agent_v0` from inside it.

In [25]:
import sys

sys.path.insert(0, "../phase-0-foundation")

from agent_v0 import SYSTEM_PROMPT, ask, call_llm, new_conversation, parse_tool_call, read_file

import inspect

A quick check that it's the Phase 0 agent we know: its prompt still describes one hand-written tool.

In [26]:
print(SYSTEM_PROMPT)

You are a helpful coding assistant.

You have ONE tool:
  read_file(filename) - returns the contents of a text file.

To use it, reply with exactly one line and nothing else:
tool: read_file({"filename": "path/to/file"})

After you receive a tool_result(...) message, use it to continue the task.
If you don't need the tool, just answer normally.


## 1. Tool registry: game plan


*Agreed 2026-10-31. The sub-steps below are built one at a time, in order.*

### What we're doing and why

Phase 0 has one tool, `read_file`, and its name is written by hand in **three places** in `agent_v0/loop.py`:

| Place | Lines | What's hard-coded |
|---|---|---|
| The system prompt | [`loop.py:10-14`](../phase-0-foundation/agent_v0/loop.py) | "You have ONE tool: read_file(filename) ..." |
| The parser | [`loop.py:22`](../phase-0-foundation/agent_v0/loop.py) | `prefix = "tool: read_file("` |
| The loop | [`loop.py:51-52`](../phase-0-foundation/agent_v0/loop.py) | `result = read_file(args.get("filename", ""))` |

So adding a second tool would mean editing all three. The roadmap's goal for this step ([`ROADMAP.md:181`](../ROADMAP.md)): *"Describe tools automatically from their code; add a tool without touching the loop."* It comes first in Phase 1 because steps 2, 4 and 6 all add tools ([`ROADMAP.md:70`](../ROADMAP.md)).

### Evidence: how real agents do it

The notebook is how *we* do it. This table is how everyone else does: small-scale projects first, then the five big open-source industry agents (Gemini CLI, Codex CLI, opencode, OpenHands, Aider). Source code read on 2026-10-31; all links are pinned to the commit read. **This is a snapshot:** these repos change daily, so the table describes them as of that date and isn't updated afterwards.

| Tier | Agent | How it registers tools | Source |
|---|---|---|---|
| Small scale | **learn-claude-code** (lesson s02) | Two separate pieces: a hand-written list of tool descriptions (`TOOLS`) and a dictionary from name to function (`TOOL_HANDLERS`). The loop calls `TOOL_HANDLERS[name](**args)`; an unknown name returns `"Unknown: <name>"`. Its own summary: *"the loop stays the same; only tool registration and dispatch grow."* | [`s02_tool_use/code.py:138-156, 180-181`](https://github.com/shareAI-lab/learn-claude-code/blob/7226f555d48900076eba126acf37c9ed5f002b5d/s02_tool_use/code.py#L138-L181) |
| Small scale | **ghuntley/how-to-build-a-coding-agent** (Go) | One bundle per tool, `ToolDefinition`: name, description, input schema and function together. The description is hand-written; the input schema is **generated** from a type. The loop searches the tool list by name; an unknown name returns "tool not found". | [`read.go:253-271, 147-170`](https://github.com/ghuntley/how-to-build-a-coding-agent/blob/fa3dd49fdddf45e92b35c66746e6bb6aac6ac27e/read.go#L253-L271) |
| Small scale | **mini-swe-agent** (text mode) | **No registry.** One action type, a shell command, pulled out of the reply with a regex. A badly formatted reply is sent back to the model as an error message. | [`actions_text.py:15-40`](https://github.com/SWE-agent/mini-swe-agent/blob/04d809ceab9df28f9adaed044884180159172930/src/minisweagent/models/utils/actions_text.py#L15-L40) |
| Industry | **Gemini CLI** (Google, TypeScript, ~109k stars) | A `ToolRegistry` class holding a map from name to tool object; `registerTool` overwrites a duplicate name with a warning. Tools come from three sources: built in, **discovered** by running a project command, and MCP servers. The list sent to the model is rebuilt from each tool's own schema, **minus tools excluded in config**; in Plan Mode it **rewrites the write and edit tools' descriptions** to "only for plans". An unknown tool returns `Tool "x" not found.` plus **"did you mean" suggestions**: the 3 closest names by edit distance. Descriptions are hand-written in separate files per model family, and **the same tool gets different descriptions for different models**: for `read_file`, the Gemini 3 set says the model "MUST use 'start_line' and 'end_line' for targeted, surgical reads"; the default set doesn't. | [`tool-registry.ts:247-294`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/tool-registry.ts#L247-L294), [`:370-530`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/tool-registry.ts#L370-L530), [`:663-718`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/tool-registry.ts#L663-L718); [`scheduler.ts:374-419`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/scheduler/scheduler.ts#L374-L419); [`tool-utils.ts:65`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/utils/tool-utils.ts#L65); [`default-legacy.ts:84-86`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/definitions/model-family-sets/default-legacy.ts#L84-L86), [`gemini-3.ts:93-95`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/definitions/model-family-sets/gemini-3.ts#L93-L95), [`resolver.ts:17-34`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/definitions/resolver.ts#L17-L34) |
| Industry | **Codex CLI** (OpenAI, Rust, ~132k stars) | A `ToolRegistry` holding an ordered map from name to tool. **Two ways in**: `register_trusted` for built-in tools and `register_external` for outside tools, which must first pass a **tool policy**. Each built-in tool's description and parameters are **hand-written in code**, and vary by situation (the shell tool adds Windows guidance; approval parameters are added to its schema). An unknown tool returns `"unsupported call: <name>"` to the model, and the failed call is **logged to telemetry** (OpenTelemetry). | [`registry.rs:428-433, 516-580`](https://github.com/openai/codex/blob/67f0994b532b8740f14b5b738af7b7b2a915b1d6/codex-rs/core/src/tools/registry.rs#L428-L580), [`:766-786, 1148-1151`](https://github.com/openai/codex/blob/67f0994b532b8740f14b5b738af7b7b2a915b1d6/codex-rs/core/src/tools/registry.rs#L766-L786); [`shell_spec.rs:14-108`](https://github.com/openai/codex/blob/67f0994b532b8740f14b5b738af7b7b2a915b1d6/codex-rs/core/src/tools/handlers/shell_spec.rs#L14-L108) |
| Industry | **opencode** (independent, TypeScript, ~224k stars) | A registry keyed by tool name, with an editor API to add, update and remove tools; each tool is **validated when it's registered** (bad names or schemas become registration errors). Before each model call it takes a **snapshot** of the tools, dropping any that the agent's **permission rules** deny, so the model never sees them. Each parameter carries its own hand-written description. An unknown tool returns *"No tool named "x" is currently available. Please use a tool from the available tool list."* It also has a **Code Mode**: instead of calling tools one at a time, the model writes a small program that calls them. | [`tool.ts:22-42`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool.ts#L22-L42), [`:265-290`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool.ts#L265-L290), [`:328-335`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool.ts#L328-L335), [`:381-414`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool.ts#L381-L414); [`tool/plugin/read.ts:17-43`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool/plugin/read.ts#L17-L43) |
| Industry | **OpenHands** (All Hands AI, Python SDK; main repo ~92k stars) | A **Python dictionary** from name to factory: `register_tool(name, factory)` takes a ready tool or a class that builds tools per conversation; a duplicate name logs a warning and overwrites. Each tool's arguments are a typed (Pydantic) class whose fields carry hand-written descriptions, and the terminal tool has **separate Unix and Windows descriptions**. Ships **model-specific tool sets** (e.g. Gemini-style file tools). An unknown tool returns *"Tool 'x' not found. Available: ..."*; **wrong arguments return a validation error to the model**, and it even tries to detect and repair some malformed calls. | [`tool/registry.py:34, 130-175`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-sdk/openhands/sdk/tool/registry.py#L130-L175); [`agent/agent.py:1414-1417, 1481`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-sdk/openhands/sdk/agent/agent.py#L1414-L1481); [`terminal/definition.py:56-102`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-tools/openhands/tools/terminal/definition.py#L56-L102); [`tools/gemini/`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-tools/openhands/tools/gemini) |
| Industry | **Aider** (Python, ~50k stars) | **No tool registry, by design.** The model doesn't call tools to edit; it writes edits in a **text format** (whole files, search/replace blocks, unified diffs), which Aider parses and applies. The "registry" is a list of these *edit formats*, picked by name; an unknown one is an error listing the valid formats. Function-calling versions exist in the code but are **commented out** of the active list. | [`coders/__init__.py:16-34`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/__init__.py#L16-L34); [`base_coder.py:61, 190-191`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/base_coder.py#L190-L191); [`editblock_coder.py:18-41, 439`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/editblock_coder.py#L18-L41); [`wholefile_func_coder.py:8-25`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/wholefile_func_coder.py#L8-L25) |

**What the six agents with a tool registry agree on, and so what this plan follows:**
- **A lookup from tool name to tool**, and the loop finds tools by name (1b, 1e). Every agent with a registry does this, from 150-line tutorials to Gemini CLI, Codex, opencode and OpenHands.
- **An unknown tool is an error the model sees**, not a crash (1e). The better ones tell the model what *is* available (OpenHands lists the tools, opencode points to the tool list, Gemini CLI suggests the closest names). This also matches Phase 0's own rule: `read_file` returns errors instead of raising them.

**Where they differ:**
- **Where descriptions come from.** All six agents with a registry hand-write their descriptions. Only ghuntley and OpenHands generate anything automatically, and only the argument schema, from a typed class. Two industry agents go further the other way: Gemini CLI and OpenHands keep **different descriptions or tool sets for different models**, treating descriptions as prompt text to tune, not just documentation. This bears on decision 1 below, and on Phase 2's "same harness, different models" ([`ROADMAP.md:90`](../ROADMAP.md)).
- **Industry tool lists aren't fixed.** In Gemini CLI, Codex and opencode, which tools the model sees depends on config, policy, permissions or mode, and tools can arrive at runtime (discovery, MCP, external tools). The small projects have a fixed list. We'll meet the same pressures later: the approval prompt is step 5, MCP is step 14 ([`ROADMAP.md:100`](../ROADMAP.md)).
- **Bad arguments.** OpenHands validates arguments against the tool's schema and returns the error to the model, and tries to repair some malformed calls. This bears on decision 3 below.
- **Unknown-tool errors get smarter at scale.** The small projects just say "unknown"; the industry agents list or suggest alternatives, and Codex records the failure in telemetry, which is the kind of record Phase 2 (measurement) builds.
- **Text protocol vs native tool calling.** All the registry agents use **native tool calling** (the API receives the tool list and returns structured requests). We keep our text protocol for now; native tool calling is step 12, and comparing the two is step 13 ([`ROADMAP.md:98-99`](../ROADMAP.md)).
- **No registry at all** is a real design too, in two very different ways: mini-swe-agent has one shell tool (the roadmap's "Bash-only agent", Later, [`ROADMAP.md:187`](../ROADMAP.md)), and **Aider**, a major production agent, edits through text formats instead of tool calls, with its function-calling coders commented out. Text protocols like ours aren't just a teaching shortcut.

Not covered: Claude Code and Cursor aren't open source, so they'd appear only through their public docs; not read for this step.

### Sub-steps

From the problem, the evidence and the decisions below. Each one is a separate turn: built, run and discussed before the next.

| # | Sub-step | Replaces | Based on |
|---|---|---|---|
| 1a | **Look at the problem.** Print Phase 0's parser and `ask()` (its prompt is already printed in the setup above), and find the three hard-coded places. Also see what happens today to a malformed tool call. | — | The problem table above; decision 3 |
| 1b | **The registry.** One place that lists the tools: a dictionary from tool name to function. Register Phase 0's `read_file`, unchanged. Read what the registry can tell us about a tool: its parameters (from the signature) and its description (the docstring). | — | All six agents with a registry keep a lookup from name to tool; decision 1 |
| 1c | **Describe tools from the registry.** Build the prompt's tool list from what's registered: parameters from each signature, description from each docstring. Explain the format with a generic pattern **plus one concrete example line, generated from the first registered tool.** | `loop.py:10-14` | Gemini CLI builds the model's tool list from the registry; decisions 1 and 2 |
| 1d | **Parse any tool name, with three outcomes.** Phase 0's parser with two changes: (1) the prefix becomes `"tool: "` and the parser reads the tool's name, so any tool works; (2) where Phase 0 returned `None` on bad JSON, it raises a format error, plus one new check for arguments that aren't a JSON object. Outcomes: no tool call (a final answer), a valid call, or a malformed call (a format error). | `loop.py:22-29` | mini-swe-agent sends format errors back to the model; decision 3 |
| 1e | **Run a tool by name.** Look it up and call it. An unknown tool, or wrong argument names, come back as an error result listing what's available or expected (the expected parameters come from the signature). Try both failure cases. | `loop.py:51-52` | All six agents with a registry return unknown tools as errors; OpenHands lists available tools and returns argument errors; decisions 1 and 3 |
| 1f | **The new loop.** Phase 0's `ask()`, rewritten to use 1c–1e. A format error goes back to the model as a message, instead of ending the conversation. | `ask()` | learn-claude-code: *"the loop stays the same; only tool registration and dispatch grow"*; decision 3 |
| 1g | **Try it.** Phase 0's demo question, then print the conversation. Then register a tiny throwaway tool and check that it appears in the prompt and can be called, with no other change. | — | "Done when" checks 1 and 2 |

Not in this step, though industry agents do them: tools filtered by config or mode, tools added at runtime, "did you mean" suggestions, telemetry, per-model descriptions, and full type checking of arguments. These belong to later steps (approval, MCP, measurement, native tool calling).

### Decisions (agreed 2026-10-31)

1. **Where does a tool's description come from? Parameters from the function's signature; description from its docstring.** A description has two parts with different risks. Parameter names written separately can drift from the function, and then every call fails; generated from the signature, they can't. The description stays hand-written, as in all six agents with a registry, but it lives in the function itself, so each tool is described in one place. This is the hybrid ghuntley and OpenHands use (schema generated, description hand-written). It also prepares step 12: native tool calling needs a typed schema, which our type hints (`filename: str`) can provide. Not done: per-model descriptions like Gemini CLI's; with one model there's nothing to tune yet.
2. **Concrete example or generic pattern? Both:** a generic pattern explains the format, plus one real example line generated from the first registered tool, so nothing is hard-coded. A small model likely copies a concrete line more reliably; our only evidence so far is 1 bad run in 4 with a generic pattern alone, in code since removed. To watch: with several tools, the model may favor the one in the example. **This is measurable**, so it's a candidate Phase 2 experiment: example vs no example, on the same tasks.
3. **Arguments that aren't a JSON object? The parser tells three outcomes apart, and a malformed call goes back to the model as a format error.** Looking into this found a bigger problem in Phase 0: on bad JSON its parser returns `None` ([`loop.py:26-29`](../phase-0-foundation/agent_v0/loop.py)), which `ask()` treats as "no tool requested", so it **prints the broken call as the final answer and stops** ([`loop.py:46-49`](../phase-0-foundation/agent_v0/loop.py)). The model never learns it made a mistake. Now: the **parser** handles format (no call / valid call / malformed call), and **`run_tool`** handles content (unknown tool, wrong argument names). Not done: a validation library such as Pydantic; real type checking belongs to step 12, with real schemas.

### Done when

- With only `read_file` registered, the agent answers Phase 0's question the same way as before.
- Adding a tool needs only registering a function: no change to the prompt, parser or loop.
- Asking for an unknown tool, or passing wrong arguments, gives the model an error result instead of crashing.
- A malformed tool call goes back to the model as a format error, instead of ending the conversation.


### 1a. Look at the problem

Before changing anything, read Phase 0's code. `inspect.getsource` prints the source of an imported function, so we can read `agent_v0` without leaving the notebook. Look for every place that names `read_file`.

In [27]:
import inspect

print(inspect.getsource(parse_tool_call))
print(inspect.getsource(ask))

def parse_tool_call(reply: str):
    """Return the tool's arguments if the reply asks for read_file, else None."""
    prefix = "tool: read_file("
    for line in reply.splitlines():
        line = line.strip()
        if line.startswith(prefix) and line.endswith(")"):
            try:
                return json.loads(line[len(prefix):-1])
            except json.JSONDecodeError:
                return None
    return None

def ask(user_message: str, conversation: list, max_steps: int = 5) -> str:
    """Run the agent loop on one user message. Appends to `conversation`."""
    conversation.append({"role": "user", "content": user_message})

    for step in range(max_steps):
        reply = call_llm(conversation)                          # 1. ask the model
        conversation.append({"role": "assistant", "content": reply})

        args = parse_tool_call(reply)                           # 2. did it ask for a tool?
        if args is None:
            print("Assistant:", reply)         

Besides the system prompt printed in the setup, two places hard-code `read_file`:

- **The parser** only recognizes one tool: `prefix = "tool: read_file("`.
- **The loop** always calls one function: `result = read_file(args.get("filename", ""))`.

Now the hidden problem (decision 3). The parser returns `None` for two very different replies: a normal answer, and a tool call it couldn't read. Watch:

In [28]:
normal_answer = "It will print hello world."
broken_call = 'tool: read_file({filename: "sandbox/hello.py"})'   # missing quotes: not valid JSON

print(parse_tool_call(normal_answer))
print(parse_tool_call(broken_call))

None
None


Both return `None`. And in `ask()`, `None` means "no tool requested, this is the final answer". So a broken tool call is **printed as the agent's answer and the conversation stops**. The model never finds out it made a mistake.

**What 1a found:** three places to replace (prompt, parser, loop), and one silent failure to fix. Steps 1b–1f handle these.

### 1b. The registry

One place that lists the tools: a dictionary from **tool name** to **the Python function**. Registering a tool just adds it to the dictionary, under the function's own name.

We register Phase 0's `read_file` exactly as it is: nothing about the function changes.

In [29]:
TOOLS = {}


def register(function):
    """Make a Python function available to the model as a tool."""
    TOOLS[function.__name__] = function


register(read_file)
TOOLS

{'read_file': <function agent_v0.tools.read_file(filename: str) -> dict>}

Decision 1: everything the model needs to know about a tool comes from the function itself. Python can read it for us:

- the **parameters**, from the function's signature (`inspect.signature`);
- the **description**, from its docstring (the `"""..."""` text under `def`).

In [30]:
print(inspect.signature(read_file))                 # the full signature
print(list(inspect.signature(read_file).parameters))  # just the parameter names
print(read_file.__doc__)                             # the docstring

(filename: str) -> dict
['filename']
Read a text file and return its contents.


In [31]:
TOOLS

{'read_file': <function agent_v0.tools.read_file(filename: str) -> dict>}

### 1c. Describe tools from the registry

Now build the prompt from the registry instead of typing it. Three small pieces:

1. **One line per tool**: name, parameters and description.
2. **One concrete example line** (decision 2), generated from a registered tool, so it's real but not hard-coded.
3. **The prompt**, which puts them together with the same protocol as Phase 0.

In [32]:
def describe_tool(function) -> str:
    """One line for the prompt: name(parameters) - description."""
    params = ", ".join(inspect.signature(function).parameters)
    return f"  {function.__name__}({params}) - {function.__doc__}"


print(describe_tool(read_file))

  read_file(filename) - Read a text file and return its contents.


In [33]:
import json


def example_call(function) -> str:
    """A real example of the tool-call line, with placeholder values."""
    args = {name: "..." for name in inspect.signature(function).parameters}
    return f"tool: {function.__name__}({json.dumps(args)})"


print(example_call(read_file))

tool: read_file({"filename": "..."})


In [34]:
def build_system_prompt(tools: dict) -> str:
    tool_lines = "\n".join(describe_tool(f) for f in tools.values())
    first_tool = next(iter(tools.values()))
    return f"""You are a helpful coding assistant.

You have these tools:
{tool_lines}

To use a tool, reply with exactly one line and nothing else:
tool: tool_name({{"argument_name": "value"}})

For example:
{example_call(first_tool)}

After you receive a tool_result(...) message, use it to continue the task.
If you don't need a tool, just answer normally."""


print(build_system_prompt(TOOLS))

You are a helpful coding assistant.

You have these tools:
  read_file(filename) - Read a text file and return its contents.

To use a tool, reply with exactly one line and nothing else:
tool: tool_name({"argument_name": "value"})

For example:
tool: read_file({"filename": "..."})

After you receive a tool_result(...) message, use it to continue the task.
If you don't need a tool, just answer normally.


Compare with Phase 0's prompt in the setup above:

| | Phase 0 | Now |
|---|---|---|
| Tool line | Typed by hand: `read_file(filename) - returns the contents of a text file.` | Generated from the function's name, signature and docstring |
| Format | One exact line for `read_file` | A generic pattern **plus** one example generated from a registered tool |
| Adding a tool | Edit the prompt | Register the function; its line appears automatically |

Nothing uses this prompt yet. The loop (1f) will.

### 1d. Parse any tool name, with three outcomes

This is Phase 0's parser (printed in 1a) with **two changes**, and nothing else:

1. **Any tool name, not just `read_file`.** The prefix shrinks from `"tool: read_file("` to `"tool: "`, and the parser reads the name that follows. It returns `(tool_name, args)` instead of just `args`.
2. **Errors instead of a silent `None`.** Where Phase 0 returned `None` on bad JSON, this raises a `ToolFormatError` with a reason. One new check also catches arguments that aren't a JSON object (the bare-string case).

So the three outcomes are: no tool line → `None` (a final answer); a valid line → `(tool_name, args)`; an unreadable line → `ToolFormatError`. Raising an error follows mini-swe-agent, which raises a `FormatError` for badly formatted replies. The loop (1f) will catch it and send the message back to the model.

In [35]:
import json


class ToolFormatError(Exception):
    """The reply tried to call a tool, but the arguments couldn't be read."""


def parse_tool_call(reply: str):
    """Return (tool_name, args), None if there's no tool call, or raise ToolFormatError."""
    prefix = "tool: "                                                   # changed: any tool
    for line in reply.splitlines():
        line = line.strip()
        if line.startswith(prefix) and line.endswith(")") and "(" in line:
            name, _, raw_args = line[len(prefix):-1].partition("(")      # changed: read the name
            try:
                args = json.loads(raw_args)
            except json.JSONDecodeError:
                raise ToolFormatError(f"The arguments are not valid JSON: {raw_args}")   # changed
            if not isinstance(args, dict):                                               # new
                raise ToolFormatError(f'The arguments must be a JSON object, like {{"filename": "a.py"}}. Got: {raw_args}')
            return name, args                                           # changed: name too
    return None

Try it on the replies that matter, including the broken call from 1a and the bare-string call seen in an earlier test run:

In [36]:
replies = [
    "It will print hello world.",                              # no tool call
    'tool: read_file({"filename": "sandbox/hello.py"})',        # valid
    'tool: list_files({"folder": "."})',                        # valid, any tool name
    'tool: read_file({filename: "sandbox/hello.py"})',          # broken JSON (from 1a)
    'tool: read_file("sandbox/hello.py")',                      # a bare string, not an object
]

for reply in replies:
    try:
        print("OK:   ", parse_tool_call(reply))
    except ToolFormatError as e:
        print("ERROR:", e)

OK:    None
OK:    ('read_file', {'filename': 'sandbox/hello.py'})
OK:    ('list_files', {'folder': '.'})
ERROR: The arguments are not valid JSON: {filename: "sandbox/hello.py"}
ERROR: The arguments must be a JSON object, like {"filename": "a.py"}. Got: "sandbox/hello.py"


What to notice:

- The broken call from 1a is now an **error with a reason**, not a silent `None`.
- The bare string `"sandbox/hello.py"` is caught, with a message showing the right shape.
- `list_files` is accepted, though it doesn't exist. The parser only checks the **format**; whether the tool exists is checked when running it (1e).

### 1e. Run a tool by name

Phase 0 ran its tool with one line (printed in 1a):

```python
result = read_file(args.get("filename", ""))
```

`run_tool(name, args)` replaces it: look the tool up in the registry and call it with the model's arguments. `**args` turns `{"filename": "a.py"}` into `filename="a.py"`. Two things can go wrong, and both come back as an `{"error": ...}` result the model can read, like `read_file`'s own errors:

1. **An unknown tool.** The error lists the tools that do exist, as OpenHands does.
2. **Wrong arguments.** `inspect.signature(function).bind(**args)` checks the arguments against the function's parameters *without running it*. The error shows the arguments given and the ones expected.

(Phase 0's `args.get("filename", "")` quietly hid wrong arguments: a call with `{"path": ...}` became `read_file("")` and a confusing "File not found".)

In [37]:
def run_tool(name: str, args: dict) -> dict:
    """Call a registered tool. Unknown tools and wrong arguments become error results."""
    if name not in TOOLS:
        return {"error": f"Unknown tool: {name}. Available tools: {', '.join(TOOLS)}"}

    function = TOOLS[name]
    try:
        inspect.signature(function).bind(**args)
    except TypeError:
        expected = ", ".join(inspect.signature(function).parameters)
        return {"error": f"Wrong arguments for {name}. Got: {', '.join(args) or 'none'}. Expected: {expected}"}

    return function(**args)

Try the working case and both failure cases. The file is Phase 0's `hello.py`, which already exists.

In [38]:
print(run_tool("read_file", {"filename": "../phase-0-foundation/sandbox/hello.py"}))   # works
print(run_tool("list_files", {"folder": "."}))                                       # unknown tool
print(run_tool("read_file", {"path": "a.py"}))                                       # wrong argument name
print(run_tool("read_file", {}))                                                     # missing argument

{'filename': '../phase-0-foundation/sandbox/hello.py', 'content': 'name = "world"\nprint(f"hello {name}")\n'}
{'error': 'Unknown tool: list_files. Available tools: read_file'}
{'error': 'Wrong arguments for read_file. Got: path. Expected: filename'}
{'error': 'Wrong arguments for read_file. Got: none. Expected: filename'}


What to notice:

- Every outcome is a dict the loop can send back to the model; nothing crashes.
- Each error tells the model what to do instead: which tools exist, or which parameters are expected.
- Not handled here, same as Phase 0: a tool that crashes *while running*. That's each tool's own job, as `read_file` does for missing files.

### 1f. The new loop

Phase 0's `new_conversation()` and `ask()` (printed in 1a), copied exactly, with only these changes (marked in the code):

1. **`new_conversation()`** uses the prompt generated from the registry (1c) instead of the hand-written `SYSTEM_PROMPT`.
2. **The parser call** is wrapped in `try`: a `ToolFormatError` (1d) goes back to the model as a `tool_result` with the error, and the loop continues, instead of the broken call ending the conversation.
3. **Which tool:** the parser now returns `(name, args)`, so the loop unpacks both.
4. **Running it:** `run_tool(name, args)` (1e) replaces `read_file(args.get("filename", ""))`.

Everything else, including the step limit and how results are sent back, is unchanged.

In [39]:
def new_conversation() -> list:
    """A fresh conversation: just the system prompt."""
    return [{"role": "system", "content": build_system_prompt(TOOLS)}]   # changed: generated prompt


def ask(user_message: str, conversation: list, max_steps: int = 5) -> str:
    """Run the agent loop on one user message. Appends to `conversation`."""
    conversation.append({"role": "user", "content": user_message})

    for step in range(max_steps):
        reply = call_llm(conversation)                          # 1. ask the model
        conversation.append({"role": "assistant", "content": reply})

        try:                                                    # changed: the parser can raise
            call = parse_tool_call(reply)                       # 2. did it ask for a tool?
        except ToolFormatError as e:                            # new: a broken call goes back to the model
            print(f"  [step {step + 1}] format error: {e}")
            conversation.append(
                {"role": "user", "content": f"tool_result({json.dumps({'error': str(e)})})"}
            )
            continue
        if call is None:
            print("Assistant:", reply)                          # 3a. no -> final answer
            return reply

        name, args = call                                       # changed: which tool
        print(f"  [step {step + 1}] model asked for {name}({args})")
        result = run_tool(name, args)                           # 3b. changed: any registered tool
        conversation.append(                                    # 4. send the result back
            {"role": "user", "content": f"tool_result({json.dumps(result)})"}
        )

    print("Stopped: too many tool calls.")

Nothing runs yet: `ask()` calls the model, so it's tried in 1g. Note that a format error counts toward `max_steps`, so a model that keeps sending broken calls still stops after 5 steps.

### 1g. Try it

Two checks from "done when":

1. With only `read_file` registered, the agent answers Phase 0's question the same way as before.
2. A new tool works after just registering it: no change to the prompt, parser or loop.

First, the same safe playground as Phase 0: a `sandbox/` folder in this phase's folder.

In [40]:
from pathlib import Path

sandbox = Path("sandbox")
sandbox.mkdir(exist_ok=True)
(sandbox / "hello.py").write_text('name = "world"\nprint(f"hello {name}")\n');

**Check 1:** Phase 0's demo question.

In [41]:
conversation = new_conversation()
ask("What will sandbox/hello.py print when I run it?", conversation);

  [step 1] model asked for read_file({'filename': 'sandbox/hello.py'})
Assistant: When you run sandbox/hello.py, it will print:

```
hello world
```

The script defines a variable `name` with the value "world", then uses an f-string to print "hello" followed by the value of the `name` variable.


Look inside, as in Phase 0: every message, including exactly what the model wrote when it asked for a tool.

In [42]:
for i, m in enumerate(conversation):
    print(f"{i}  {m['role']:9}  {m['content'][:80]!r}")

0  system     'You are a helpful coding assistant.\n\nYou have these tools:\n  read_file(filename)'
1  user       'What will sandbox/hello.py print when I run it?'
2  assistant  "I'll check the contents of the sandbox/hello.py file to see what it will print w"
3  user       'tool_result({"filename": "sandbox/hello.py", "content": "name = \\"world\\"\\nprint'
4  assistant  'When you run sandbox/hello.py, it will print:\n\n```\nhello world\n```\n\nThe script d'


**Check 2:** a throwaway tool. Write a function, register it, and nothing else.

In [43]:
def count_lines(filename: str) -> dict:
    """Count the lines in a text file."""
    path = Path(filename)
    if not path.exists():
        return {"error": f"File not found: {filename}"}
    return {"filename": filename, "lines": len(path.read_text().splitlines())}


register(count_lines)
print(build_system_prompt(TOOLS))

You are a helpful coding assistant.

You have these tools:
  read_file(filename) - Read a text file and return its contents.
  count_lines(filename) - Count the lines in a text file.

To use a tool, reply with exactly one line and nothing else:
tool: tool_name({"argument_name": "value"})

For example:
tool: read_file({"filename": "..."})

After you receive a tool_result(...) message, use it to continue the task.
If you don't need a tool, just answer normally.


Its line appeared in the prompt on its own. Now ask a question that needs it:

In [44]:
conversation = new_conversation()
ask("How many lines are in sandbox/hello.py?", conversation);

  [step 1] model asked for count_lines({'filename': 'sandbox/hello.py'})
Assistant: There are 2 lines in sandbox/hello.py.


It was throwaway, so remove it again; section 2 adds the real tools.

In [45]:
del TOOLS["count_lines"]
list(TOOLS)

['read_file']

## 2. List and find files: game plan


*Agreed 2026-11-12. The sub-steps below are built one at a time, in order.*

### What we're doing and why

The agent can read a file only if it's given the file's path. It can't see what files exist, so a request like "what's in the config file?" fails unless we name the path. The roadmap ([`ROADMAP.md:71`](../ROADMAP.md)): *"The agent has to explore a project before it can change it."*

With the registry from step 1, a new tool is just a function plus `register()`. So this step is about **what** the tool does, not how to wire it in.

**Scope: find files by name or path.** The tool takes a pattern (such as `*` or `**/*.py`) and returns matching file paths. The agent can then read what it found with `read_file`.

| Not in this step | Where it is |
|---|---|
| Searching *inside* files ("which file prints hello?") | **Search contents**, a Later item ([`ROADMAP.md:184`](../ROADMAP.md)), not scheduled in any phase. Step 6's shell will let the agent run `grep` itself. |
| A summary of the codebase put in the prompt up front | **Repo map**, a Later item ([`ROADMAP.md:212`](../ROADMAP.md)), not scheduled |
| Keeping the tool inside the project folder | **Step 3**, the folder boundary ([`ROADMAP.md:72`](../ROADMAP.md)) |
| A shell (`ls`, `find`) | **Step 6** ([`ROADMAP.md:75`](../ROADMAP.md)) |

### Evidence: how real agents do it

Source code read on 2026-10-31, links pinned to the commit read. **This is a snapshot:** these repos change daily, so the tables describe them as of that date and aren't updated afterwards.

**By question: what they agree on, where they differ, and what we follow** (the sources are in the per-agent table below):

| Question | What real agents do | For us |
|---|---|---|
| 1. A dedicated tool for finding files? | **Most have one:** learn-claude-code, ghuntley, Gemini CLI, opencode. OpenHands has one but leaves it out of its default tool set. **Two use the shell instead:** Codex (`rg --files`) and mini-swe-agent (`ls`, `find`). **Aider doesn't let the model explore at all:** it puts a repo map in the prompt. | A dedicated tool: we have no shell yet, and the repo map is out of scope. |
| 2. What does the tool take? | **A pattern (glob)** is the most common: learn-claude-code, Gemini CLI, opencode, OpenHands. **A folder to list:** Gemini CLI's `ls` (alongside its glob), ghuntley's `list_files`, opencode's `read` (which also reads folders). | **Decision 1** |
| 3. Are results capped? | **Yes, almost everywhere,** with a message telling the model to narrow its pattern: learn-claude-code (200), OpenHands (100), opencode. Only ghuntley has no limit. | **Decision 2** |
| 4. What gets skipped? | Gemini CLI skips what `.gitignore` lists; opencode skips hidden files; ghuntley skips one folder. | **Decision 2** |
| 5. What order? | Most recently modified first: Gemini CLI, OpenHands. Otherwise alphabetical. | **Decision 3** |

**One agent at a time** (the record behind the table above):

| Tier | Agent | How the model explores a project | Source |
|---|---|---|---|
| Small scale | **learn-claude-code** | One `glob` tool (plus the shell). Results capped at 200, with *"narrow the pattern"* when there are more; matches outside the workspace are dropped. | [`s02_tool_use/code.py:120-133`](https://github.com/shareAI-lab/learn-claude-code/blob/7226f555d48900076eba126acf37c9ed5f002b5d/s02_tool_use/code.py#L120-L133) |
| Small scale | **ghuntley** | One `list_files` tool that walks the whole tree from a folder, marking folders with `/`. No limit; it skips only one folder (`.devenv`). | [`list_files.go:271-333`](https://github.com/ghuntley/how-to-build-a-coding-agent/blob/fa3dd49fdddf45e92b35c66746e6bb6aac6ac27e/list_files.go#L271-L333) |
| Small scale | **mini-swe-agent** | No file tools: the model runs `ls` and `find` through its one shell tool. | (its single shell action; see step 1's table) |
| Industry | **Gemini CLI** | Two tools: `ls` (one folder) and `glob` (a pattern). `ls` respects `.gitignore` by default and refuses paths outside the workspace. `glob` lists **recently modified files first**, then the rest alphabetically. | [`ls.ts:46-50, 172-175, 218-225`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/ls.ts#L46-L225); [`glob.ts:44-70`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/glob.ts#L44-L70) |
| Industry | **Codex CLI** | **No listing tool at all.** Its model tools are the shell and `apply_patch`, and the prompt says: *"When searching for text or files, prefer using `rg` or `rg --files`."* | [`gpt-5.2-codex_prompt.md:5`](https://github.com/openai/codex/blob/67f0994b532b8740f14b5b738af7b7b2a915b1d6/codex-rs/core/gpt-5.2-codex_prompt.md#L5); [`tools/handlers/`](https://github.com/openai/codex/blob/67f0994b532b8740f14b5b738af7b7b2a915b1d6/codex-rs/core/src/tools/handlers) |
| Industry | **opencode** | **No separate listing tool:** `read` reads a directory as well as a file, one entry per line. Plus a `glob` tool with a result limit; when it's hit, the model is told to use a more specific path or pattern. Hidden files are excluded by default. | [`read.ts:18-44`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool/plugin/read.ts#L18-L44); [`glob.ts:18-39`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool/plugin/glob.ts#L18-L39) |
| Industry | **OpenHands** | Has a `glob` tool (sorted by modification time, first 100 results, with a "narrow your search" hint), but its **default tool set doesn't include it**: by default the model explores through the terminal. | [`glob/definition.py:42-55`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-tools/openhands/tools/glob/definition.py#L42-L55); [`preset/default.py:56-58`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-tools/openhands/tools/preset/default.py#L56-L58) |
| Industry | **Aider** | **No exploring at all: it gives the model a map up front.** A "repo map" summarizes the repository with tree-sitter, ranks files and symbols with PageRank, fits it into a token budget (1,024 by default), and puts it in the prompt as *"summaries of some files present in my git repository"*. | [`repomap.py:42-120, 365-422`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/repomap.py#L42-L120); [`base_prompts.py:45-48`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/base_prompts.py#L45-L48) |

**Cross-check with a survey.** The *Harness Engineering* survey ([Barbaste et al., July 2026](https://arxiv.org/abs/2609.00006); see [RELATED-WORK.md](../RELATED-WORK.md)) reads 11 agents' source and lists their code-search tools in its Table 13 (p.55). It agrees with the tables above: OpenHands has glob/grep tools plus the terminal, and opencode's glob caps results at 100. It's a July snapshot without line numbers, so our tables, read from source, are still the evidence.

**A counterpoint on order.** The survey recommends starting with a shell (Recommendation 3, p.68): *"Begin with just a bash tool"*, and add glob only *"when bash+find/rg feels awkward"*. Our roadmap puts the shell later on purpose: the approval prompt (step 5) is *"needed before shell access"* ([`ROADMAP.md:74`](../ROADMAP.md)).

### Sub-steps (proposed)

| # | Sub-step | Based on |
|---|---|---|
| 2a | **Add the tool.** Write `find_files(pattern)`, register it, and check it offline on a small sandbox project with files in subfolders (e.g. a `config.json` one level down), including the cap. | The registry from step 1; questions 1–5 above |
| 2b | **Try it.** Ask a question about a file by name without giving its path (e.g. *"what does the config file contain?"*), and watch the agent find the file, then read it. | "Done when" below |

### Decisions (agreed 2026-11-12)

1. **One tool: `find_files(pattern)`** (glob), like learn-claude-code. `"*"` lists the top folder, `"**/*.py"` finds Python files anywhere, and listing a folder is just the pattern `folder/*`. The fewest tools; the model has to know glob syntax, which models generally do. Not done: a separate `list_files(folder)` tool like Gemini CLI's `ls`.
2. **Cap at 100 results, with a "narrow your pattern" message** when there are more (as OpenHands and opencode do), and **skip hidden files and folders** such as `.git` (as opencode does), which would otherwise flood any real project. Not done: `.gitignore` support (Gemini CLI).
3. **Alphabetical order:** simple and predictable. Not done: most recently modified first (Gemini CLI, OpenHands), which matters more in big projects; we can revisit.

### Done when

- Asked about a file by name, without its path, the agent finds it with the new tool, then reads it.
- A pattern that matches too many files returns the first 100 plus a message to narrow it, not a flood.
- Adding the tool needed nothing but `register()`: step 1 still holds.


In [54]:
# 2a. Add the tool: find_files(pattern), registered like any other tool (capped at a limit, alphabetical, hidden files skipped)
import shutil

MAX_RESULTS = 10


def find_files(pattern: str) -> dict:
    """Find files by name with a glob pattern, e.g. "*.py" or "**/*.json". Patterns only search the top folder; use **/ to search subfolders."""   # changed: after 2b
    matches = sorted(                                                   # alphabetical
        str(path) for path in Path(".").glob(pattern)
        if path.is_file() and not any(part.startswith(".") for part in path.parts)   # skip hidden files and folders
    )
    if not matches:                                                     # new: after 2b, an empty result says what to try
        return {"files": [], "note": "No files match. Use **/ to search subfolders."}
    if len(matches) > MAX_RESULTS:                                      # cap, and say so
        return {"files": matches[:MAX_RESULTS],
                "note": f"Showing {MAX_RESULTS} of {len(matches)} files. Narrow your pattern."}
    return {"files": matches}


register(find_files)
print(describe_tool(find_files), "\n")

# A small project to search: files in subfolders, plus a hidden folder that should never show up
for name, text in {
    "sandbox/app/config.json": '{"greeting": "hello", "debug": false}\n',
    "sandbox/app/main.py": 'print("main")\n',
    "sandbox/tests/test_main.py": "def test_main():\n    assert True\n",
    "sandbox/.git/HEAD": "ref: refs/heads/main\n",
}.items():
    Path(name).parent.mkdir(parents=True, exist_ok=True)
    Path(name).write_text(text)

# Checks, through run_tool as the model would call it
print(run_tool("find_files", {"pattern": "**/config.json"}))        # by name, anywhere
print(run_tool("find_files", {"pattern": "sandbox/**/*"}))          # the whole project; .git/HEAD is skipped
print(run_tool("find_files", {"pattern": "**/*.rs"}))               # no match

# The cap: 150 files, then clean up so the sandbox stays small
Path("sandbox/many").mkdir(exist_ok=True)
for i in range(150):
    (Path("sandbox/many") / f"file_{i:03}.txt").write_text("")
result = run_tool("find_files", {"pattern": "sandbox/many/*"})
print(len(result["files"]), "files,", result["note"])
shutil.rmtree("sandbox/many")


  find_files(pattern) - Find files by name with a glob pattern, e.g. "*.py" or "**/*.json". Patterns only search the top folder; use **/ to search subfolders. 

{'files': ['sandbox/app/config.json']}
{'files': ['sandbox/app/config.json', 'sandbox/app/main.py', 'sandbox/hello.py', 'sandbox/tests/test_main.py']}
{'files': [], 'note': 'No files match. Use **/ to search subfolders.'}
10 files, Showing 10 of 150 files. Narrow your pattern.


In [47]:
# 2b. Try it: ask about a file by name, without giving its path
conversation = new_conversation()
ask("What does the config file in this project contain?", conversation)

print()
for i, m in enumerate(conversation[1:], start=1):   # everything after the system prompt
    print(f"{i}  {m['role']:9}  {m['content'][:100]!r}")


  [step 1] model asked for find_files({'pattern': 'config*'})
  [step 2] model asked for find_files({'pattern': '**/config*'})
  [step 3] model asked for read_file({'filename': 'sandbox/app/config.json'})
Assistant: The config file at `sandbox/app/config.json` contains the following JSON configuration:

```json
{
  "greeting": "hello",
  "debug": false
}
```

This is a simple configuration file with two settings:
1. A greeting message set to "hello"
2. A debug flag set to false

1  user       'What does the config file in this project contain?'
2  assistant  "I'll help you find and examine the config file in this project. Let me first search for files that m"
3  user       'tool_result({"files": [], "note": "No files match. Use **/ to search subfolders."})'
4  assistant  'Let me search for configuration files in all subfolders.\n\ntool: find_files({"pattern": "**/config*"}'
5  user       'tool_result({"files": ["sandbox/app/config.json"]})'
6  assistant  'Great! I found a config file a

## 3. Folder boundary



*Agreed 2026-11-24. Full plan and research: [phase_1_research.md, step 3](phase_1_research.md#step-3-folder-boundary).*

**Goal:** limit the files the agent can read or change to one project folder.

**Breakdown:** the agent reads files (today) and will write them (next step), through our own file tools (today) or terminal commands (later in Phase 1). The only live part today: **reading, through our own tools**.

**Chosen:** a check inside our tool code, as learn-claude-code and Gemini CLI do: resolve the path, and **refuse** it unless it's inside **`sandbox/`**. Dropped: containers and OS sandboxes (out of scope for this project) and protecting `.git` or `.env` inside the project. Terminal commands will be able to get around this check; there, the approval prompt (you OK each command) is the guard.

**Steps** (the two cells below):
1. **As-is:** print `read_file` and `find_files`, call them on paths outside the project, and ask the model to read `../ROADMAP.md`. It reads it.
2. **Build the check:** `inside_project(path)` resolves the path (following symlinks) and checks it's under `sandbox/`; `read_file` and `find_files` call it.
3. **Test it:** without the model, a normal path, `../`, `/etc/hosts` and a symlink pointing out; then the same model question as the as-is cell. This time it's refused.

**Done when:** `read_file` refuses `../`, absolute and symlink paths with an error naming the folder; `find_files` never returns a path outside it; asked for an outside file, the agent gets the refusal, and we record what it does next.

In [52]:
# 3. As-is: neither file tool checks where a path points
print(inspect.getsource(read_file))    # Phase 0's version (agent_v0)
print(inspect.getsource(find_files))   # section 2's version

# Called directly, the way the loop calls them
# print(run_tool("read_file", {"filename": "../ROADMAP.md"})["content"][:100], "...\n")
# print(run_tool("find_files", {"pattern": "../*"}), "\n")   # empty, but by accident: the hidden-name filter also skips ".."

# # The same through the model. The next cell asks this exact question again, after the check is built.
OUTSIDE_QUESTION = "Read ../ROADMAP.md and tell me in one sentence what this project is."
conversation = new_conversation()
ask(OUTSIDE_QUESTION, conversation);

def read_file(filename: str) -> dict:
    """Read a text file and return its contents."""
    path = Path(filename)
    if not path.exists():
        return {"error": f"File not found: {filename}"}
    return {"filename": filename, "content": path.read_text()}

def find_files(pattern: str) -> dict:
    """Find files by name with a glob pattern, e.g. "*.py" or "**/*.json". Patterns only search the top folder; use **/ to search subfolders."""   # changed: after 2b
    matches = sorted(                                                   # decision 3: alphabetical
        str(path) for path in Path(".").glob(pattern)
        if path.is_file() and not any(part.startswith(".") for part in path.parts)   # decision 2: skip hidden
    )
    if not matches:                                                     # new: after 2b, an empty result says what to try
        return {"files": [], "note": "No files match. Use **/ to search subfolders."}
    if len(matches) > MAX_RESULTS:                     

In [55]:
# 3. The folder boundary: both file tools check the path before touching anything; outside paths are refused, and the boundary is sandbox/
# Execution:
# 1. ROOT, the boundary: the sandbox/ folder.
# 2. inside_project(path): resolve the path (following symlinks) and check it's still under ROOT.
# 3. read_file and find_files call it: read_file refuses outside paths, find_files drops outside matches. Paths are now relative to sandbox/.
ROOT = Path("sandbox").resolve()                                    # the boundary: the project the agent works on


def inside_project(path) -> bool:
    """True if the path, with symlinks followed, ends up inside ROOT."""
    return (ROOT / path).resolve().is_relative_to(ROOT)            # learn-claude-code's safe_path, in one line


def read_file(filename: str) -> dict:
    """Read a text file and return its contents."""
    if not inside_project(filename):                                # new: the boundary; refuse outside paths
        return {"error": f"{filename} is outside the project folder. Only files inside it can be read, with paths relative to it."}
    path = ROOT / filename                                          # changed: relative to the project, not the notebook
    if not path.exists():
        return {"error": f"File not found: {filename}"}
    return {"filename": filename, "content": path.read_text()}


def find_files(pattern: str) -> dict:
    """Find files by name with a glob pattern, e.g. "*.py" or "**/*.json". Patterns only search the top folder; use **/ to search subfolders."""
    matches = sorted(
        str(path.relative_to(ROOT)) for path in ROOT.glob(pattern)     # changed: search the project, paths relative to it
        if path.is_file() and inside_project(path)                     # new: the boundary, which also catches symlinks
        and not any(part.startswith(".") for part in path.relative_to(ROOT).parts)
    )
    if not matches:
        return {"files": [], "note": "No files match. Use **/ to search subfolders."}
    if len(matches) > MAX_RESULTS:
        return {"files": matches[:MAX_RESULTS],
                "note": f"Showing {MAX_RESULTS} of {len(matches)} files. Narrow your pattern."}
    return {"files": matches}


register(read_file)                                                 # replaces the old entries in TOOLS
register(find_files)

# Tests, through run_tool as the model would call them. A symlink inside the project points outside it.
Path("sandbox/link").symlink_to(Path("../phase-0-foundation").resolve())
for name, args in [
    ("read_file", {"filename": "hello.py"}),                 # inside: works
    ("find_files", {"pattern": "**/*.py"}),                   # inside: works, paths relative to the project
    ("read_file", {"filename": "../ROADMAP.md"}),            # climbs out
    ("read_file", {"filename": "/etc/hosts"}),               # absolute path
    ("read_file", {"filename": "link/phase_0_README.md"}),   # through the symlink
    ("find_files", {"pattern": "link/*"}),                    # through the symlink
]:
    print(f"{name}({args}) ->", str(run_tool(name, args))[:120])
Path("sandbox/link").unlink()

# The same question as the as-is cell, now with the boundary
print()
conversation = new_conversation()
ask(OUTSIDE_QUESTION, conversation);

read_file({'filename': 'hello.py'}) -> {'filename': 'hello.py', 'content': 'name = "world"\nprint(f"hello {name}")\n'}
find_files({'pattern': '**/*.py'}) -> {'files': ['app/main.py', 'hello.py', 'tests/test_main.py']}
read_file({'filename': '../ROADMAP.md'}) -> {'error': '../ROADMAP.md is outside the project folder. Only files inside it can be read, with paths relative to it.'}
read_file({'filename': '/etc/hosts'}) -> {'error': '/etc/hosts is outside the project folder. Only files inside it can be read, with paths relative to it.'}
read_file({'filename': 'link/phase_0_README.md'}) -> {'error': 'link/phase_0_README.md is outside the project folder. Only files inside it can be read, with paths relative t
find_files({'pattern': 'link/*'}) -> {'files': [], 'note': 'No files match. Use **/ to search subfolders.'}

  [step 1] model asked for read_file({'filename': '../ROADMAP.md'})
Assistant: I cannot read the ROADMAP.md file because it's located outside the project folder. Please move it in

## 4. Write and edit files



*Agreed 2026-11-26. Full plan and research: [phase_1_research.md, step 4](phase_1_research.md#step-4-write-and-edit-files).*

**Goal:** let the agent change a file inside the project: the "fixes it" part of Phase 1's goal.

**Breakdown:** only **changing part of an existing file** is needed (not creating, rewriting, deleting or renaming). This step handles old text that's missing or appears twice; the folder check from step 3 handles paths outside `sandbox/`; the approval prompt, later, handles changes you didn't want.

**Chosen:** `edit_file(filename, old_text, new_text)`, find-and-replace as most agents do: the old text must match exactly and appear once, otherwise a clear error says what to do. Dropped: patch formats, editing through the shell, forgiving matches.

**Decisions (agreed):** (A) find-and-replace, not whole files; "whole file vs find-and-replace on our model" is queued for the Phase 2 benchmark. (B) no tool to create or overwrite files for now.

**Steps** (the cell below):
1. **Build** `edit_file(filename, old_text, new_text)` and register it.
2. **Test without the model** on a small `greetings.py`: a normal edit, text not found, text found twice, a path outside the project.
3. **Test with the model:** ask it to change `farewell`, and watch it read the file, then edit it.

**Done when:** asked for a small change, the agent reads the file, edits it, and the file changes; missing text, repeated text and outside paths are refused with a helpful error.

In [56]:
# 4. The edit tool: find-and-replace (not whole-file rewrites); old_text must match exactly and appear once; no tool to create files yet
# Execution:
# 1. edit_file(filename, old_text, new_text): replaces old_text with new_text, inside sandbox/ only (it calls inside_project).
# 2. Two refusals with advice: old_text not found, or found more than once.
# 3. register(edit_file): the tool appears in the model's prompt automatically.
def edit_file(filename: str, old_text: str, new_text: str) -> dict:
    """Replace old_text with new_text in a file. Read the file first: old_text must be copied exactly (spaces included) and appear once."""
    if not inside_project(filename):                                    # the folder boundary from step 3
        return {"error": f"{filename} is outside the project folder. Only files inside it can be edited, with paths relative to it."}
    path = ROOT / filename
    if not path.exists():
        return {"error": f"File not found: {filename}"}
    text = path.read_text()
    count = text.count(old_text)
    if count == 0:                                                      # error 1: the text isn't there
        return {"error": f"old_text was not found in {filename}. Read the file and copy the text exactly, including spaces."}
    if count > 1:                                                       # error 2: ambiguous
        return {"error": f"old_text appears {count} times in {filename}. Include more of the surrounding code so it appears once."}
    path.write_text(text.replace(old_text, new_text))
    return {"result": f"Edited {filename}."}


register(edit_file)
print(describe_tool(edit_file), "\n")

# A small file to edit; removed at the end of the cell
demo = ROOT / "greetings.py"
original = 'def greet(name):\n    return "Hello, " + name\n\n\ndef farewell(name):\n    return "Goodbye, " + name\n'
demo.write_text(original)

# Tests, through run_tool as the model would call it
for args in [
    {"filename": "greetings.py", "old_text": '"Hello, "', "new_text": '"Hi, "'},     # works
    {"filename": "greetings.py", "old_text": '"Hey, "', "new_text": '"Hi, "'},       # not found
    {"filename": "greetings.py", "old_text": "    return", "new_text": "  return"},   # appears twice
    {"filename": "../ROADMAP.md", "old_text": "#", "new_text": "!"},                  # outside the project
]:
    print(f"edit_file({args}) ->", run_tool("edit_file", args))
print(demo.read_text())

# With the model: a small change, starting from the original file
demo.write_text(original)
conversation = new_conversation()
ask("In greetings.py, change farewell so it says 'See you, ' instead of 'Goodbye, '.", conversation)
print("\n" + demo.read_text())
demo.unlink()

  edit_file(filename, old_text, new_text) - Replace old_text with new_text in a file. Read the file first: old_text must be copied exactly (spaces included) and appear once. 

edit_file({'filename': 'greetings.py', 'old_text': '"Hello, "', 'new_text': '"Hi, "'}) -> {'result': 'Edited greetings.py.'}
edit_file({'filename': 'greetings.py', 'old_text': '"Hey, "', 'new_text': '"Hi, "'}) -> {'error': 'old_text was not found in greetings.py. Read the file and copy the text exactly, including spaces.'}
edit_file({'filename': 'greetings.py', 'old_text': '    return', 'new_text': '  return'}) -> {'error': 'old_text appears 2 times in greetings.py. Include more of the surrounding code so it appears once.'}
edit_file({'filename': '../ROADMAP.md', 'old_text': '#', 'new_text': '!'}) -> {'error': '../ROADMAP.md is outside the project folder. Only files inside it can be edited, with paths relative to it.'}
def greet(name):
    return "Hi, " + name


def farewell(name):
    return "Goodbye, " + name



## 5. Approval prompt



*Agreed 2026-11-26. Full plan and research: [phase_1_research.md, step 5](phase_1_research.md#step-5-approval-prompt).*

**Goal:** before the agent changes anything, show you the action and wait for your yes or no: the "asks before anything risky" part of Phase 1's goal, needed before the agent can run terminal commands.

**Breakdown:** tools that only look (`find_files`, `read_file`) are never asked; tools that change files (`edit_file`) are; the terminal-command tool will be, once built. On "yes" the tool runs; on "no" it doesn't, and the model is told so it doesn't assume the change happened.

**Chosen:** one check in the agent loop, between the model's request and running the tool, for tools listed in `RISKY_TOOLS`. Dropped: "yes for the whole session", modes like auto-edit or yolo, rule lists, and letting the model rate its own risk (OpenHands does that; noted for later).

**Decisions (agreed):** (A) ask before every edit. (B) Enter to allow, or type a reason to refuse, which the model receives (mini-swe-agent's prompt).

**Steps** (the cell below):
1. **Build** `RISKY_TOOLS` and `approve()`, and add one check to `ask()` before a risky tool runs.
2. **Test with the model, twice, on the same edit request. You answer the prompt:** press Enter (the file changes), then type a reason (it doesn't, and the model gets your reason).

**Done when:** an edit waits for your answer; yes applies it; a typed reason refuses it, leaves the file unchanged and reaches the model; reading never prompts.

In [57]:
# 5. The approval prompt: the loop asks you before every edit; press Enter to allow, or type a reason to refuse
# Execution:
# 1. A list of risky tools: RISKY_TOOLS = {"edit_file"}.
# 2. A small approve() function that shows the prompt and reads your answer.
# 3. One added check in ask(), right before a tool runs. If the tool is on the list, it asks you first. On a refusal, the tool doesn't run, and the model gets a result with your reason instead.
RISKY_TOOLS = {"edit_file"}                                             # every edit asks; the terminal tool joins later


def approve(name: str) -> str | None:
    """Ask the user. Returns None if allowed, or the reason they gave for refusing."""
    answer = input(f"  Allow {name}? Press Enter to allow, or type why not: ").strip()
    return None if answer.lower() in ("", "y", "yes") else answer      # anything else is a refusal, with your text as the reason


def ask(user_message: str, conversation: list, max_steps: int = 5) -> str:
    """Run the agent loop on one user message. Appends to `conversation`."""
    conversation.append({"role": "user", "content": user_message})

    for step in range(max_steps):
        reply = call_llm(conversation)                          # 1. ask the model
        conversation.append({"role": "assistant", "content": reply})

        try:
            call = parse_tool_call(reply)                       # 2. did it ask for a tool?
        except ToolFormatError as e:
            print(f"  [step {step + 1}] format error: {e}")
            conversation.append(
                {"role": "user", "content": f"tool_result({json.dumps({'error': str(e)})})"}
            )
            continue
        if call is None:
            print("Assistant:", reply)                          # 3a. no -> final answer
            return reply

        name, args = call
        print(f"  [step {step + 1}] model asked for {name}({args})")
        refusal = approve(name) if name in RISKY_TOOLS else None            # new: ask before risky tools
        if refusal is None:
            result = run_tool(name, args)                       # 3b. yes -> run it
        else:
            result = {"error": f"The user refused this action. Their reason: {refusal}"}   # new: tell the model why
        conversation.append(                                    # 4. send the result back
            {"role": "user", "content": f"tool_result({json.dumps(result)})"}
        )

    print("Stopped: too many tool calls.")


# Try it twice on the same request. You answer the prompts in the notebook.
demo = ROOT / "greetings.py"
original = 'def greet(name):\n    return "Hello, " + name\n\n\ndef farewell(name):\n    return "Goodbye, " + name\n'
request = "In greetings.py, change farewell so it says 'See you, ' instead of 'Goodbye, '."

print("Run 1: press Enter to allow the edit.")
demo.write_text(original)
ask(request, new_conversation())
print("\n" + demo.read_text())

print("Run 2: refuse with a reason, e.g. \"no, change greet to say 'Hi, ' instead\".")
demo.write_text(original)
ask(request, new_conversation())
print("\n" + demo.read_text())
demo.unlink()

Run 1: press Enter to allow the edit.
  [step 1] model asked for read_file({'filename': 'greetings.py'})
  [step 2] model asked for edit_file({'filename': 'greetings.py', 'old_text': 'def farewell(name):\n    return "Goodbye, " + name', 'new_text': 'def farewell(name):\n    return "See you, " + name'})


  Allow edit_file? Press Enter to allow, or type why not:  y


Assistant: I've updated the farewell function in greetings.py to say "See you, " instead of "Goodbye, ". The change has been made successfully.

def greet(name):
    return "Hello, " + name


def farewell(name):
    return "See you, " + name

Run 2: refuse with a reason, e.g. "no, change greet to say 'Hi, ' instead".
  [step 1] model asked for read_file({'filename': 'greetings.py'})
  [step 2] model asked for edit_file({'filename': 'greetings.py', 'old_text': 'def farewell(name):\n    return "Goodbye, " + name', 'new_text': 'def farewell(name):\n    return "See you, " + name'})


  Allow edit_file? Press Enter to allow, or type why not:  no. i think this is too risky.


Assistant: I understand that you're hesitant about making this change. Let me explain what this modification would do:

The change would simply replace the word "Goodbye" with "See you" in the farewell function, so when the function is called, it would say "See you, [name]" instead of "Goodbye, [name]".

If you're concerned about making changes to the code, that's completely understandable. Would you like me to suggest an alternative approach, or perhaps explain more about why this change might be useful?

def greet(name):
    return "Hello, " + name


def farewell(name):
    return "Goodbye, " + name



## 6. Run shell commands



*Agreed 2026-11-26. Full plan and research: [phase_1_research.md, step 6](phase_1_research.md#step-6-run-shell-commands).*

**Goal:** let the agent run a terminal command such as `pytest` and see its output: how it sees a test fail, and confirms a fix.

**Breakdown:** a command can look, run code, change files, or reach outside the project; our code limits none of that, so your yes or no (the approval prompt) is the only guard. The tool itself must handle where it runs, how long, what comes back, and how much.

**Chosen:** `run_command(command)`: a fresh process in `sandbox/`, a 30 s limit, the exit code plus output and errors, cut past 10,000 characters (keeping start and end), the notebook's own Python first on the `PATH`, and on the risky list. Dropped: a long-lived terminal, background commands, deny lists, OS sandboxes.

**Decisions (agreed):** (A) approve every command. (B) a fixed 30 s limit; the model can't change it.

**Steps** (the cell below):
1. **Build** `run_command(command)` and add it to `RISKY_TOOLS`.
2. **Test without the model:** `ls`, which Python runs, a failing command, and a command stopped by the time limit.
3. **Test with the model. You answer the prompts:** run the tests on a tiny project with a bug (press Enter); then `cat ../ROADMAP.md`, a command outside the project that the folder check can't stop.

**Done when:** after your yes, the agent runs `pytest`, sees the failing test and says what fails; failures and timeouts come back clearly; every command waits for your yes, including one that reaches outside the project.

In [58]:
# 6. The shell tool: run one command in sandbox/; every command needs your yes, and the time limit is a fixed 30 s


# Execution:
# 1. run_command(command): a fresh process in sandbox/, stopped after 30 s, with the notebook's Python first on the PATH.
# 2. It returns the exit code and the output (errors included), keeping the start and end past 10,000 characters.
# 3. It joins RISKY_TOOLS, so every command waits for your yes in ask().


import os
import shutil
import subprocess
import sys

COMMAND_TIMEOUT = 30                                                    # fixed; the model can't change it
MAX_OUTPUT = 10_000


def run_command(command: str) -> dict:
    """Run a shell command in the project folder and return its exit code and output. Use it to run tests, e.g. "python -m pytest"."""
    env = os.environ | {"PATH": os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]}   # the notebook's Python first
    try:
        r = subprocess.run(command, shell=True, cwd=ROOT, env=env, timeout=COMMAND_TIMEOUT,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)          # errors mixed into the output
    except subprocess.TimeoutExpired:
        return {"error": f"The command took longer than {COMMAND_TIMEOUT} seconds and was stopped."}
    output = r.stdout
    if len(output) > MAX_OUTPUT:                                        # keep the start and the end
        half = MAX_OUTPUT // 2
        output = output[:half] + f"\n... ({len(output) - MAX_OUTPUT} characters cut) ...\n" + output[-half:]
    return {"exit_code": r.returncode, "output": output}


register(run_command)
RISKY_TOOLS.add("run_command")                                          # every command needs your yes
print(describe_tool(run_command), "\n")

# Tests, through run_tool (no approval prompt here: that lives in ask())
for command in ["ls", "python -c 'import sys; print(sys.executable)'", "ls no_such_file"]:
    print(f"run_command({command!r}) ->", run_tool("run_command", {"command": command}))
COMMAND_TIMEOUT = 1                                                     # a short limit, just for this test
print("run_command('sleep 3') ->", run_tool("run_command", {"command": "sleep 3"}))
COMMAND_TIMEOUT = 30

# A tiny project with one bug and its failing test; removed at the end of the cell
(ROOT / "calc.py").write_text("def add(a, b):\n    return a - b\n")
(ROOT / "tests" / "test_calc.py").write_text("from calc import add\n\n\ndef test_add():\n    assert add(2, 3) == 5\n")

print("\nRun 1: press Enter to allow the command. If it then asks to edit calc.py, answer as you like.")
ask("Run the tests and tell me what fails.", new_conversation())

print("\nRun 2: the approval prompt shows a command that reads outside the project. Enter lets it through; the folder check can't stop it.")
ask("Show me what's in ../ROADMAP.md using cat. Just the first 5 lines.", new_conversation())

(ROOT / "calc.py").unlink()
(ROOT / "tests" / "test_calc.py").unlink()
for cache in ["__pycache__", "tests/__pycache__", ".pytest_cache"]:              # what pytest leaves behind
    shutil.rmtree(ROOT / cache, ignore_errors=True)

  run_command(command) - Run a shell command in the project folder and return its exit code and output. Use it to run tests, e.g. "python -m pytest". 

run_command('ls') -> {'exit_code': 0, 'output': '\x1bapp\x1b[m\x1b[m\nhello.py\n\x1btests\x1b[m\x1b[m\n'}
run_command("python -c 'import sys; print(sys.executable)'") -> {'exit_code': 0, 'output': '/opt/anaconda3/envs/agents/bin/python\n'}
run_command('ls no_such_file') -> {'exit_code': 1, 'output': 'ls: no_such_file: No such file or directory\n'}
run_command('sleep 3') -> {'error': 'The command took longer than 1 seconds and was stopped.'}

Run 1: press Enter to allow the command. If it then asks to edit calc.py, answer as you like.
  [step 1] model asked for run_command({'command': 'python -m pytest'})


  Allow run_command? Press Enter to allow, or type why not:  


Assistant: The test `test_add` in `tests/test_calc.py` is failing. The assertion `assert add(2, 3) == 5` is failing because the function `add(2, 3)` is returning `-1` instead of the expected `5`. 

The other test in `tests/test_main.py` is passing. 

Would you like me to look into the implementation of the `add` function to understand why it's returning `-1`?

Run 2: the approval prompt shows a command that reads outside the project. Enter lets it through; the folder check can't stop it.
  [step 1] model asked for run_command({'command': 'cat ../ROADMAP.md | head -5'})


  Allow run_command? Press Enter to allow, or type why not:  no


  [step 2] model asked for read_file({'filename': '../ROADMAP.md'})
  [step 3] model asked for find_files({'pattern': '*ROADMAP*.md'})
  [step 4] model asked for find_files({'pattern': '**/*roadmap*.md'})
  [step 5] model asked for find_files({'pattern': '**/*ROADMAP*'})
Stopped: too many tool calls.


## 7. Verify loop



*Draft, 2026-11-26: two decisions open. Full plan and research: [phase_1_research.md, step 7](phase_1_research.md#step-7-verify-loop).*

**Goal:** after changing code, the agent runs the tests, and if they still fail, fixes again until they pass: the last part of Phase 1's goal, "runs the tests to confirm the fix".

**Breakdown:** the check exists (`pytest` through `run_command`). Missing: a reason for the agent to run it after each change (our prompt says nothing about testing) and room to repeat (our loop stops after 5 model calls, which a single fix cycle already uses up). Who decides "done" can be the model (asked in the prompt), the harness (forces the check), or a second model (judges the result).

**Proposed:** one added instruction in the system prompt: run the tests after changing code, fix and re-run until they pass, and report the command and its result. A 10-step limit for these runs. Dropped: a second model as judge (the roadmap's "goal check", later).

**Decisions:** (A) ask the model to verify in the prompt, or make the loop force it? Lean: the prompt only, and watch whether the model follows it. (B) guard against the agent editing the test instead of the code, or just watch? Lean: watch; your approval prompt shows every edit, and the cell checks the test file afterwards.

**Steps:**
1. **As-is** (the cell below): today's prompt, a tiny project with a bug and its failing test, *"The tests fail. Fix the bug."* You answer the prompts. Does it re-run the tests after editing?
2. **Build:** `build_system_prompt` plus one verify instruction.
3. **Test:** the same request with the new prompt and a 10-step limit; then check that the test file is unchanged and the tests pass. This is also Phase 1's end-to-end demo.

**Done when:** with the new prompt, the agent fixes the code, re-runs the tests, finishes only when they pass and shows the evidence; the test file is unchanged; and we can compare its behaviour with and without the instruction.